# 🧠 Oxford Parkinson's Voice Biomarkers — Model Training Pipeline
### Minor Project Part-I: Acoustic Model Architecture & Fitting
**Dataset:** Oxford Parkinson's Disease (`processed_data/cleaned_parkinsons.csv`)  
**Scope:** **Task 3 — Training Only (No Evaluation)**  
> **Note:** Per Task 3 instructions, this notebook strictly performs **dataset partitioning, acoustic feature standardization, model fitting, and binary serialization**. Test evaluation and metric computation will be conducted in the next phase.

## 1. Setup & Library Imports
Load scikit-learn estimators and serialization tools.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# Estimators & preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "processed_data", "cleaned_parkinsons.csv")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Environment initialized successfully.")
print(f"Target Directory for Saved Models: {MODEL_DIR}")

Environment initialized successfully.
Target Directory for Saved Models: C:\Users\Lakshya\OneDrive\Desktop\Minor-Project-1\saved_models


## 2. Data Loading & Partitioning
Load cleaned acoustic feature records and partition using an 80/20 stratified split ($N_{\text{train}} = 156$, $N_{\text{test}} = 39$).

In [2]:
# 2.1 Load Cleaned Data
df = pd.read_csv(DATA_PATH)
print(f"Loaded Parkinson's Cleaned Data: {df.shape[0]} voice records, {df.shape[1]} features + target")

X = df.drop(columns=['status'])
y = df['status']

# 2.2 Stratified 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Features Shape (X_train) : {X_train.shape}")
print(f"Training Labels Shape (y_train)   : {y_train.shape}")
print(f"Testing Features Shape (X_test)   : {X_test.shape} (Preserved for Task 4)")
print(f"Testing Labels Shape (y_test)     : {y_test.shape}")
print("\nTarget distribution in Training Set:")
print(y_train.value_counts().rename({0: 'Healthy Control (0)', 1: "Parkinson's (1)"}))

Loaded Parkinson's Cleaned Data: 195 voice records, 23 features + target
Training Features Shape (X_train) : (156, 22)
Training Labels Shape (y_train)   : (156,)
Testing Features Shape (X_test)   : (39, 22) (Preserved for Task 4)
Testing Labels Shape (y_test)     : (39,)

Target distribution in Training Set:
status
Parkinson's (1)        118
Healthy Control (0)     38
Name: count, dtype: int64


## 3. Acoustic Feature Standardization
Standardize continuous acoustic measurements (Frequencies, Jitter, Shimmer, HNR, PPE).

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Persist scaler
scaler_path = os.path.join(MODEL_DIR, "parkinsons_scaler.pkl")
joblib.dump(scaler, scaler_path)

print(f"[SAVED] Parkinson's StandardScaler exported to: {scaler_path}")
print(f"Features scaled: {X.columns.tolist()[:5]} ... (+{len(X.columns)-5} more)")

[SAVED] Parkinson's StandardScaler exported to: C:\Users\Lakshya\OneDrive\Desktop\Minor-Project-1\saved_models\parkinsons_scaler.pkl
Features scaled: ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)'] ... (+17 more)


## 4. Model Instantiation & Training
Train **5 distinct classification algorithms** tailored for acoustic voice biomarker detection:
1. **Linear SVM:** High-dimensional linear maximum-margin separator.
2. **RBF SVM:** Non-linear Radial Basis Function kernel mapping.
3. **Decision Tree (CART):** Threshold-based decision logic.
4. **Random Forest:** Ensemble of decision trees for robust vocal modeling.
5. **K-Nearest Neighbors (KNN):** Distance clustering in normalized acoustic space.

In [4]:
# 4.1 Define Models
models = {
    'Linear SVM': SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    'RBF SVM': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

trained_models = {}

print("=" * 75)
print(" EXECUTING MODEL TRAINING ON PARKINSON'S COHORT (N = 156)")
print("=" * 75)

# 4.2 Model Fitting
for name, model in models.items():
    print(f"\nFitting [{name}] ...")
    if 'SVM' in name or 'Neighbors' in name:
        model.fit(X_train_scaled, y_train)
        input_type = "Scaled Acoustic Features"
    else:
        model.fit(X_train, y_train)
        input_type = "Raw Acoustic Features"
        
    trained_models[name] = model
    print(f"  ✓ Training Complete. Fitted on: {input_type}")
    if hasattr(model, 'n_support_'):
        print(f"  ✓ Total Support Vectors: {sum(model.n_support_)}")
    if hasattr(model, 'n_features_in_'):
        print(f"  ✓ Input Features: {model.n_features_in_}")

print("\n" + "=" * 75)
print(" ALL 5 PARKINSON'S MODELS SUCCESSFULLY TRAINED!")
print("=" * 75)

 EXECUTING MODEL TRAINING ON PARKINSON'S COHORT (N = 156)

Fitting [Linear SVM] ...
  ✓ Training Complete. Fitted on: Scaled Acoustic Features
  ✓ Total Support Vectors: 53
  ✓ Input Features: 22

Fitting [RBF SVM] ...
  ✓ Training Complete. Fitted on: Scaled Acoustic Features
  ✓ Total Support Vectors: 79
  ✓ Input Features: 22

Fitting [Decision Tree] ...
  ✓ Training Complete. Fitted on: Raw Acoustic Features
  ✓ Input Features: 22

Fitting [Random Forest] ...
  ✓ Training Complete. Fitted on: Raw Acoustic Features
  ✓ Input Features: 22

Fitting [K-Nearest Neighbors] ...
  ✓ Training Complete. Fitted on: Scaled Acoustic Features
  ✓ Input Features: 22

 ALL 5 PARKINSON'S MODELS SUCCESSFULLY TRAINED!


## 5. Model Serialization & Persistence
Export trained model binaries to `saved_models/`.

In [5]:
file_mapping = {
    'Linear SVM': 'parkinsons_linear_svm.pkl',
    'RBF SVM': 'parkinsons_rbf_svm.pkl',
    'Decision Tree': 'parkinsons_decision_tree.pkl',
    'Random Forest': 'parkinsons_random_forest.pkl',
    'K-Nearest Neighbors': 'parkinsons_knn.pkl'
}

print("--- Serializing Trained Parkinson's Models to Disk ---")
saved_catalog = []

for name, model in trained_models.items():
    filename = file_mapping[name]
    save_path = os.path.join(MODEL_DIR, filename)
    joblib.dump(model, save_path)
    file_size_kb = os.path.getsize(save_path) / 1024
    
    saved_catalog.append({
        'Model Name': name,
        'File Name': filename,
        'File Size (KB)': f"{file_size_kb:.2f} KB",
        'Training Status': 'Trained & Serialized'
    })

catalog_df = pd.DataFrame(saved_catalog)
display(catalog_df)
print("\nAll models and scalers are persisted and ready for evaluation in Task 4.")

--- Serializing Trained Parkinson's Models to Disk ---


,Model Name,File Name,File Size (KB),Training Status
0,Linear SVM,parkinsons_linear_svm.pkl,11.79 KB,Trained & Serialized
1,RBF SVM,parkinsons_rbf_svm.pkl,16.76 KB,Trained & Serialized
2,Decision Tree,parkinsons_decision_tree.pkl,3.38 KB,Trained & Serialized
3,Random Forest,parkinsons_random_forest.pkl,219.68 KB,Trained & Serialized
4,K-Nearest Neighbors,parkinsons_knn.pkl,28.82 KB,Trained & Serialized



All models and scalers are persisted and ready for evaluation in Task 4.
